# NovelMate — Full-Stack Machine Learning Project

This notebook trains **10 machine-learning models** for book-rating prediction, saves them in `models/`, and launches the complete FastAPI + HTML/CSS/JavaScript application inside Google Colab.

## Run order
1. Run the setup cell to clone the GitHub repository.
2. Run the training cell to clean `data/books.csv` and train all models.
3. Run the application cell to open NovelMate in the notebook.

**Dataset:** Goodbooks-10k `books.csv`. The models use publication year, edition count, ratings count, review count, and language. Rating-distribution columns are deliberately excluded to avoid target leakage.


## 1. Connect to the project
Run this cell after uploading the complete project folder to your GitHub repository. If you have not pushed it yet, upload/extract the project ZIP in Colab and set `PROJECT_DIR` to that folder.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPOSITORY = "https://github.com/asadahmed-gif/NovelMate_FullStack_ML.git"
PROJECT_DIR = Path("/content/NovelMate_FullStack_ML")

if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", REPOSITORY, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
assert (PROJECT_DIR / "data" / "books.csv").exists(), "books.csv is missing from data/. Upload the complete project to GitHub."
print("Project folder:", PROJECT_DIR)
print("Dataset found:", (PROJECT_DIR / "data" / "books.csv").stat().st_size, "bytes")


## 2. Install the required libraries
The same `requirements.txt` is used for Colab and local/GitHub setup.

In [ ]:
%pip install -q -r requirements.txt
print("Dependencies installed successfully.")


## 3. Train and evaluate all models
This runs preprocessing, an 80/20 train-test split, and all regression and classification models. It saves the fitted preprocessing + model pipelines, so the backend can accept raw form values.

In [ ]:
import train_models
train_models.main()


## 4. Start the NovelMate website
This launches the FastAPI backend. The interactive website will appear below. Keep this cell running while using the app.

In [ ]:
import threading, time, requests, uvicorn
from google.colab import output

# Stop a previous server instance if this cell is run again.
try:
    server.should_exit = True
    time.sleep(1)
except NameError:
    pass

config = uvicorn.Config("app:app", host="0.0.0.0", port=8000, log_level="warning")
server = uvicorn.Server(config)
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()

for attempt in range(60):
    try:
        health = requests.get("http://127.0.0.1:8000/health", timeout=2)
        if health.ok:
            break
    except requests.RequestException:
        time.sleep(1)
else:
    raise RuntimeError("The NovelMate server did not start. Check the output above.")

info = requests.get("http://127.0.0.1:8000/api/info").json()
print("Backend status:", health.json())
print("Regression models:", len(info["models"]["regression"]))
print("Classification models:", len(info["models"]["classification"]))
output.serve_kernel_port_as_iframe(8000, height=1050)


## 5. Test the backend API
This sends a sample book record to both comparison endpoints. A successful response should contain predictions from all trained models.

In [ ]:
sample_book = {
    "original_publication_year": 2000,
    "books_count": 50,
    "ratings_count": 50000,
    "work_ratings_count": 60000,
    "work_text_reviews_count": 3000,
    "language_code": "eng"
}

for task in ["regression", "classification"]:
    response = requests.post(f"http://127.0.0.1:8000/api/compare/{task}", json=sample_book)
    print(task, "status:", response.status_code)
    if response.ok:
        for result in response.json()["results"]:
            print(" -", result["name"], "=>",
                  result.get("predicted_average_rating", result.get("category")))
    else:
        print(response.text)


## 6. Download the trained project (optional)
The trained `.joblib` files are stored in `models/`. This cell creates a ZIP of the complete project, including trained models, that can be downloaded and submitted or uploaded to GitHub.

In [ ]:
import shutil
from google.colab import files

zip_file = shutil.make_archive("/content/NovelMate_Professional", "zip", root_dir=str(PROJECT_DIR))
print("Created:", zip_file)
files.download(zip_file)
